<a href="https://colab.research.google.com/github/Maclenn77/big-data-team/blob/main/proyecto-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto Etapa 2. Selección de técnica de muestreo para la construcción de muestra inicial**

**Equipo 04**

**Daniela Montserrat Enríquez Ballesteros (A01797865)

**Humberto Alejandro Espinola Gonzalez (A01840066)

**Juan Paulo Pérez Tejada Ladrón de Guevara (A01797972)

**Mónica Raquel Saucedo Alcalá (A01797710)

## Descripción de la base de datos

### Contexto

La Comisión de Taxis y Limusinas (TLC) de la Ciudad de Nueva York (NYC) almacena datos de todos sus taxis, disponibles gratuitamente en su sitio oficial. Puedes acceder a ellos [aquí](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

- En este dataset se consideran únicamente los datos de **Taxis Amarillos**, correspondientes a **enero de 2015 y enero-marzo de 2016**, disponibles en kaggle en [este enlace](https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data).


## Atributos

| Nombre del Campo | Descripción |
|---|---|
| VendorID | Código que identifica al proveedor TPEP que generó el registro.<br><br>1. Creative Mobile Technologies<br>2. VeriFone Inc. |
| tpep_pickup_datetime | Fecha y hora en que se activó el taxímetro. |
| tpep_dropoff_datetime | Fecha y hora en que se desactivó el taxímetro. |
| Passenger_count | Número de pasajeros en el vehículo. Valor ingresado por el conductor. |
| Trip_distance | Distancia recorrida en millas, reportada por el taxímetro. |
| Pickup_longitude | Longitud del punto de recogida. |
| Pickup_latitude | Latitud del punto de recogida. |
| RateCodeID | Código de tarifa vigente al finalizar el viaje.<br><br>1. Tarifa estándar<br>2. JFK<br>3. Newark<br>4. Nassau o Westchester<br>5. Tarifa negociada<br>6. Viaje compartido |
| Store_and_fwd_flag | Indica si el registro del viaje se almacenó en la memoria del vehículo antes de enviarse al proveedor (es decir, "store and forward"), debido a falta de conexión al servidor.<br>Y = viaje almacenado y reenviado<br>N = viaje no almacenado |
| Dropoff_longitude | Longitud del punto de entrega. |
| Dropoff_latitude | Latitud del punto de entrega. |
| Payment_type | Código numérico que indica el método de pago del pasajero.<br><br>1. Tarjeta de crédito<br>2. Efectivo<br>3. Sin cargo<br>4. Disputa<br>5. Desconocido<br>6. Viaje anulado |
| Fare_amount | Tarifa calculada por el taxímetro en función del tiempo y la distancia. |
| Extra | Cargos adicionales y recargos varios. Actualmente incluye los cargos de $0.50 y $1 por hora pico y servicio nocturno. |
| MTA_tax | Impuesto MTA de $0.50, aplicado automáticamente según la tarifa del taxímetro en uso. |
| Improvement_surcharge | Recargo de mejora de $0.30, aplicado al inicio del viaje. Este recargo comenzó a cobrarse en 2015. |
| Tip_amount | Monto de la propina. Este campo se completa automáticamente para propinas pagadas con tarjeta de crédito. Las propinas en efectivo no se incluyen. |
| Tolls_amount | Monto total de peajes pagados durante el viaje. |
| Total_amount | Monto total cobrado al pasajero. No incluye propinas en efectivo. |

## Inicialización y Carga de Datos

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Create SparkSession after all environment setup
spark = SparkSession.builder \
    .appName("nyc_taxi_eda") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

In [2]:
import kagglehub
# Set the path to the file you'd like to load
file_path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")

print("Path to dataset files:", file_path)

Using Colab cache for faster access to the 'nyc-yellow-taxi-trip-data' dataset.
Path to dataset files: /kaggle/input/nyc-yellow-taxi-trip-data


In [3]:
from pyspark.sql.functions import col, hour, dayofweek, when
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, FloatType, BooleanType

# Define the schema
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("pickup_longitude", FloatType(), True),
    StructField("pickup_latitude", FloatType(), True),
    StructField("RateCodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", BooleanType(), True),
    StructField("dropoff_longitude", FloatType(), True),
    StructField("dropoff_latitude", FloatType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("extra", FloatType(), True),
    StructField("mta_tax", FloatType(), True),
    StructField("tip_amount", FloatType(), True),
    StructField("tolls_amount", FloatType(), True),
    StructField("improvement_surcharge", FloatType(), True),
    StructField("total_amount", FloatType(), True)
])

# Spark reads the 4 CSVs with the predefined schema
df = spark.read.csv(
    file_path + "/*.csv",
    header=True,
    schema=schema, # Use the defined schema
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

print(f"Filas totales: {df.count():,}")
df.printSchema()

df.show(5)

Filas totales: 47,248,845
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- pickup_longitude: float (nullable = true)
 |-- pickup_latitude: float (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: boolean (nullable = true)
 |-- dropoff_longitude: float (nullable = true)
 |-- dropoff_latitude: float (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------------+--

## Creación de Variables de Caracterización

In [4]:

df_augmented = df.withColumn("hour_of_day", hour(col("tpep_pickup_datetime")))
df_augmented = df_augmented.withColumn("day_of_week", dayofweek(col("tpep_pickup_datetime")))

# fines de semana
df_augmented = df_augmented.withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1, 7]), True).otherwise(False)
)

#horas pico
df_augmented = df_augmented.withColumn(
    "is_rush_hour",
    when(col("hour_of_day").isin([7, 8, 9, 17, 18, 19]), True).otherwise(False)
)

## Extracción de Submuestras por Partición

In [9]:
particion_1 = df_augmented.filter((col("is_weekend") == False) & (col("is_rush_hour") == True))
print(f"Total de registros en Partición 1: {particion_1.count():,}")

Total de registros en Partición 1: 11,310,928


In [10]:
particion_2 = df_augmented.filter((col("is_weekend") == False) & (col("is_rush_hour") == False))
print(f"Total de registros en Partición 2: {particion_2.count():,}")

Total de registros en Partición 2: 22,292,545


In [11]:
particion_3 = df_augmented.filter((col("is_weekend") == True) & (col("is_rush_hour") == True))
print(f"Total de registros en Partición 3: {particion_3.count():,}")

Total de registros en Partición 3: 3,262,448


In [12]:
particion_4 = df_augmented.filter((col("is_weekend") == True) & (col("is_rush_hour") == False))
print(f"Total de registros en Partición 4: {particion_4.count():,}")

Total de registros en Partición 4: 10,382,924


In [13]:
spark.stop()